# Journal study: GPU runner (Google Colab)

Runs the GPU-only parts of the judge-reliability study:

* **Part A** - open-weight target models (7-14B) against the AdvBench subset, with the same settings as the CSAM-2026 local runs (seed 42, temperature 0, 320 output tokens).
* **Part B** - Llama Guard 3 8B as an additional judge over every response file.

**Before the first run**

1. `Runtime -> Change runtime type -> T4 GPU`.
2. In Google Drive create the folder `MyDrive/jbf-journal/inputs/` and upload:
   * `advbench_subset.json` (base prompts)
   * `journal_grant_signed.json` (the signed study grant)
   * the earlier `*_judged.json` response files (for Part B)
3. In Colab open **Secrets** (key icon, left sidebar), add `JBF_AUTHORIZATION_SIGNING_KEY` and enable notebook access.

Then `Runtime -> Run all`. Results and the response cache are written to `MyDrive/jbf-journal/`, and every step resumes where it stopped if the session disconnects. Models are kept on the Colab disk (not Drive) because they are far larger than the free 15 GB Drive quota; they are re-downloaded in each new session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/jbf-journal')
INPUTS = DRIVE / 'inputs'
for sub in ('outputs', 'judge_out'):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
missing = [f for f in ('advbench_subset.json', 'journal_grant_signed.json') if not (INPUTS / f).exists()]
print('missing inputs:', missing or 'none')

In [ ]:
%cd /content
!test -d jailbreak-framework || git clone -q https://github.com/aljabid/jailbreak-framework.git
%cd /content/jailbreak-framework
!git pull -q
# results and the response cache live on Drive so a new session picks them up
!rm -rf outputs research/judge_reliability/out
!ln -s /content/drive/MyDrive/jbf-journal/outputs outputs
!ln -s /content/drive/MyDrive/jbf-journal/judge_out research/judge_reliability/out

In [ ]:
!pip -q install "uv==0.11.32"
!uv sync --locked --extra reporting -q

In [ ]:
# models stay on the Colab disk: together they exceed the free Drive quota
os.environ['OLLAMA_MODELS'] = '/content/ollama-models'
# the Ollama installer needs zstd, which Colab does not ship
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -5
import shutil, subprocess, time
assert shutil.which('ollama'), 'Ollama install failed - see the output above'
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=os.environ.copy())
time.sleep(8)
!ollama --version

In [ ]:
# signing key: prefer inputs/signing_key.txt (exact bytes from the laptop .env),
# fall back to the Colab secret. The key must be the one that SIGNED the grant.
import hashlib
key_file = INPUTS / 'signing_key.txt'
if key_file.exists():
    key = key_file.read_text().strip()
else:
    from google.colab import userdata
    key = userdata.get('JBF_AUTHORIZATION_SIGNING_KEY').strip()
os.environ['JBF_AUTHORIZATION_SIGNING_KEY'] = key
print('signing key fingerprint', hashlib.sha256(key.encode()).hexdigest()[:12])
os.environ['JBF_ATTACK_BASE_PROMPTS_FILE'] = str(INPUTS / 'advbench_subset.json')
os.environ['JBF_MODEL_TEMPERATURE'] = '0'
os.environ['JBF_MODEL_MAX_TOKENS'] = '320'
os.environ['JBF_MODEL_TIMEOUT'] = '600'
os.environ['JBF_MODEL_RATE_LIMIT_DELAY'] = '0'
GRANT = str(INPUTS / 'journal_grant_signed.json')


## Part A - target models

Each model is pulled to the Colab disk, then run through the framework. `--cache` makes reruns skip finished requests. Edit `TARGETS` to change the model set.

In [ ]:
import datetime, json
grant = json.load(open(GRANT))
now = datetime.datetime.now(datetime.timezone.utc)
start = datetime.datetime.fromisoformat(grant['not_before'].replace('Z', '+00:00'))
end = datetime.datetime.fromisoformat(grant['expires_at'].replace('Z', '+00:00'))
TARGETS = ['llama3.1:8b', 'mistral:7b', 'qwen2.5:7b', 'qwen2.5:14b']
if not start <= now <= end:
    print(f'Grant not active now ({now:%Y-%m-%d %H:%M} UTC; valid {start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} UTC). Skipping Part A.')
else:
    for m in TARGETS:
        safe = m.replace(':', '_').replace('.', '_')
        out = f'outputs/adv_local_{safe}.json'
        print(f'==== {m} -> {out}')
        !ollama pull {m} > /dev/null
        !.venv/bin/python main.py run --provider local --local-mode ollama --model-name {m}             --seed 42 --concurrency 1 --cache --no-report --eval-mode keyword             --target-id journal-study --authorization-file {GRANT} -o {out}

## Part B - Llama Guard 3 8B judge

Judges the earlier response files plus the Part A outputs. Verdicts go to `jbf-journal/judge_out/*_lg3_8b.json`.

In [ ]:
import glob, shlex
os.environ['JBF_GUARD_MODEL'] = 'llama-guard3:8b'
!ollama pull llama-guard3:8b > /dev/null
files = sorted(glob.glob(str(INPUTS / '*_judged.json'))) + sorted(glob.glob('outputs/adv_local_*.json'))
print(len(files), 'response files')
ARGS = ' '.join(shlex.quote(f) for f in files)  # IPython's {expr} expansion cannot hold nested quotes
!.venv/bin/python research/judge_reliability/judge_llamaguard.py $ARGS

Done. Everything is in `MyDrive/jbf-journal/`. With Google Drive for Desktop the files also appear on the laptop for analysis.